# Brawlhalla Detector — YOLO26 + P2, 3-class schema

Trains the perception front-end for a vision-only Brawlhalla RL agent.

**Classes** (`nc: 3`, order fixed by `data.yaml` and read back from model metadata at
inference time, so it cannot drift):

| id | class | role |
|---:|---|---|
| 0 | `character` | any fighter, any legend, any side |
| 1 | `indicator_self` | the blue self-indicator triangle |
| 2 | `weapon` | ground weapons (not held ones) |

Agent identity is **geometric**: the agent is the `character` under `indicator_self`.
Adding opponents therefore costs zero new classes.

---

## What this notebook has to answer

Training a detector is the easy half. These are the questions the downstream RL code
actually needs answered, and each has a dedicated section:

1. **`indicator_self` recall** — not overall mAP. If the indicator is missed, agent
   identity is unknown and *every* relational feature is wrong that frame.
2. **Indicator→character geometry** — the association constants in
   `feature_extractor/memory/detection_schema.py` are currently guesses. Section 8
   measures them from ground truth and emits a copy-pasteable block.
3. **Separability** — is there a score threshold that cleanly separates the true
   indicator↔character pair from the wrong ones? If not, the whole scheme is shaky.
4. **Box size distributions** — `player_w/h` are observation features; their
   normalisation ranges should come from data.
5. **Split leakage** — frames extracted from video are near-duplicates. A random
   split inflates val mAP into meaninglessness.

## 0. Architecture recommendation — read before running

You proposed **YOLO26-M + P2**. My recommendation: **keep P2, drop to S**.

### P2: keep it — this is the right call

The indicator triangle is roughly **30×20 px at 1920×1080**. Letterboxed to `imgsz=960`
the 16:9 content occupies 960×540, so the triangle lands at about **15×10 px**. The
shallowest standard head (P3) has stride 8, so a 15 px object spans under two feature
cells — right at the floor of what is reliably detectable. P2 adds a stride-4 level and
is exactly the correct fix for this object.

### Size M: the binding constraint is latency, not accuracy

The detector runs **inside a 40 Hz control loop** on a 4 GB RTX 3050 Ti, sharing the GPU
with nothing else but needing to leave headroom for capture, state update and the policy.
YOLO is already the dominant per-step cost.

P2 is expensive: a stride-4 map has **4× the spatial elements** of stride-8. Stacking
that on an M backbone is the wrong place to spend the budget when the dataset is 822
images and 3 visually distinct classes.

| Config | Capacity for 822 imgs / 3 cls | Latency risk on 3050 Ti |
|---|---|---|
| `yolo26n-p2` | likely sufficient | lowest |
| **`yolo26s-p2`** | **comfortable — recommended** | **acceptable** |
| `yolo26m-p2` | over-parameterised, overfit risk | may halve your control rate |

**Train `s` first.** If `indicator_self` recall clears the gate, ship it — a bigger model
that costs you 20 Hz of control rate is a net loss for the agent. Section 10 benchmarks
latency so this is a measured decision rather than an argument.

### Other choices baked in below

- **`imgsz=960`**, not 640. Your current inference path resizes to 640×360, which shrinks
  the indicator to ~10 px. Whatever you pick here **must be mirrored in `extract.py`** —
  train/infer resolution mismatch is a silent accuracy killer.
- **`fliplr=0.5`**. All three classes are horizontally symmetric, so this is a free 2×
  on data. (Unrelated to the RL observation canonicalisation.)
- **`epochs=300, patience=60, cos_lr=True`** — appropriate for a small dataset.
- **`close_mosaic=25`** — mosaic hurts small-object localisation late in training.

## 1. Configuration — edit this cell only

In [ ]:
from pathlib import Path

# ── Dataset ──────────────────────────────────────────────────────────────────
# Replace with your Kaggle dataset slug. Expected YOLO layout:
#
#   <DATASET_ROOT>/
#     train/images/*.png   train/labels/*.txt
#     val/images/*.png     val/labels/*.txt
#     test/images/*.png    test/labels/*.txt
#
DATASET_ROOT = Path("/kaggle/input/PLACEHOLDER-brawlhalla-detection")

TRAIN_SPLIT = "train"      # <- placeholder split directory names
VAL_SPLIT   = "val"
TEST_SPLIT  = "test"

IMAGES_SUBDIR = "images"
LABELS_SUBDIR = "labels"

# ── Class schema (must match your data.yaml exactly) ─────────────────────────
CLASS_NAMES = ["character", "indicator_self", "weapon"]
CHARACTER_ID, INDICATOR_ID, WEAPON_ID = 0, 1, 2

# ── Model / training ─────────────────────────────────────────────────────────
MODEL_CFG   = "yolo26s-p2.yaml"   # see section 0; "yolo26n-p2" / "yolo26m-p2" also valid
PRETRAINED  = "yolo26s.pt"        # transfer weights; P2 head initialises fresh
IMGSZ       = 960
EPOCHS      = 300
PATIENCE    = 60
BATCH       = 16                  # drop to 8 if Kaggle OOMs at imgsz=960
SEED        = 0

PROJECT = "/kaggle/working/runs"
NAME    = "brawl_yolo26s_p2_960"

# ── Gate for indicator recall (see section 7) ────────────────────────────────
INDICATOR_RECALL_STRONG = 0.98
INDICATOR_RECALL_USABLE = 0.95

OUT = Path("/kaggle/working"); OUT.mkdir(parents=True, exist_ok=True)
print("dataset root:", DATASET_ROOT)
print("model:", MODEL_CFG, "| imgsz:", IMGSZ, "| batch:", BATCH)

## 2. Environment

**Do not let pip upgrade torch.** Kaggle ships a torch built for the GPUs it actually
hands out. A plain `pip install ultralytics` can pull a newer torch from PyPI whose
binaries no longer contain kernels for that GPU, and the failure surfaces much later as

```
AcceleratorError: CUDA error: no kernel image is available for execution on the device
```

at the first `.to(device)` — long after the install looked fine. Recent PyTorch dropped
**Pascal (sm_60)**, which is the Kaggle **P100**, so this bites hardest there.

The cell below checks the GPU *before* installing, then installs `ultralytics` with
`--no-deps` so the working torch is left alone.

In [ ]:
import torch, platform

print("python :", platform.python_version())
print("torch  :", torch.__version__, "| built for CUDA", torch.version.cuda)
assert torch.cuda.is_available(), "No GPU. Settings -> Accelerator -> GPU."

major, minor = torch.cuda.get_device_capability(0)
device_arch = f"sm_{major}{minor}"
arch_list = torch.cuda.get_arch_list()
print("gpu    :", torch.cuda.get_device_name(0), "|", device_arch)
print("kernels:", arch_list)

if device_arch not in arch_list:
    raise SystemExit(
        f"\n*** This torch build has no kernels for {device_arch}. ***\n"
        f"    Installed: {arch_list}\n\n"
        "    Fixes, in order of preference:\n"
        "      1. Settings -> Accelerator -> GPU T4 x2 (sm_75). Modern torch dropped\n"
        "         Pascal/P100 (sm_60); T4 is still supported.\n"
        "      2. If torch was upgraded by a pip install, revert it:\n"
        "           !pip install -q --force-reinstall torch==<kaggle-original> torchvision==<...>\n"
        "         then Run -> Restart session.\n"
    )

# Real kernel launch: fails here cheaply rather than 20 minutes into training.
_x = torch.randn(64, 64, device="cuda")
_ = (_x @ _x).sum().item()
print("cuda matmul OK")

In [ ]:
# --no-deps keeps pip away from torch/torchvision. ultralytics-thop is the one
# transitive dependency Kaggle images do not already carry.
!pip -q install --no-deps "ultralytics>=8.3.0" ultralytics-thop

import importlib, torch
_torch_before = torch.__version__
import ultralytics
importlib.reload(torch)
print("ultralytics :", ultralytics.__version__)
print("torch       :", torch.__version__, "(unchanged)" if torch.__version__ == _torch_before else "*** CHANGED — restart session ***")

# Confirm the GPU still works after the install.
_x = torch.randn(64, 64, device="cuda")
_ = (_x @ _x).sum().item()
print("cuda still OK after install")

## 3. Inventory and `data.yaml`

In [ ]:
import collections, yaml

def split_dir(split): return DATASET_ROOT / split
def images_dir(split): return split_dir(split) / IMAGES_SUBDIR
def labels_dir(split): return split_dir(split) / LABELS_SUBDIR

IMG_EXT = {".png", ".jpg", ".jpeg", ".bmp"}

def list_images(split):
    d = images_dir(split)
    return sorted(p for p in d.rglob("*") if p.suffix.lower() in IMG_EXT) if d.exists() else []

def label_for(img_path, split):
    return labels_dir(split) / (img_path.stem + ".txt")

def read_label(path):
    """Return list of (cls, cx, cy, w, h) in normalised YOLO format."""
    if not path.exists():
        return []
    rows = []
    for line in path.read_text().strip().splitlines():
        parts = line.split()
        if len(parts) >= 5:
            rows.append((int(float(parts[0])), *[float(v) for v in parts[1:5]]))
    return rows

inventory, total_imgs = {}, 0
for split in (TRAIN_SPLIT, VAL_SPLIT, TEST_SPLIT):
    imgs = list_images(split)
    counts, missing, empty = collections.Counter(), 0, 0
    for im in imgs:
        lp = label_for(im, split)
        if not lp.exists():
            missing += 1; continue
        rows = read_label(lp)
        if not rows:
            empty += 1
        for cls, *_ in rows:
            counts[cls] += 1
    inventory[split] = {"images": len(imgs), "counts": counts,
                        "missing_labels": missing, "empty_labels": empty}
    total_imgs += len(imgs)

print(f"{'split':<8}{'imgs':>7}{'character':>11}{'indicator':>11}{'weapon':>9}{'no-label':>10}{'empty':>8}")
for split, info in inventory.items():
    c = info["counts"]
    print(f"{split:<8}{info['images']:>7}{c[CHARACTER_ID]:>11}{c[INDICATOR_ID]:>11}"
          f"{c[WEAPON_ID]:>9}{info['missing_labels']:>10}{info['empty_labels']:>8}")
print(f"\ntotal images: {total_imgs}")

data_yaml = OUT / "data.yaml"
yaml.safe_dump({
    "path": str(DATASET_ROOT),
    "train": f"{TRAIN_SPLIT}/{IMAGES_SUBDIR}",
    "val":   f"{VAL_SPLIT}/{IMAGES_SUBDIR}",
    "test":  f"{TEST_SPLIT}/{IMAGES_SUBDIR}",
    "nc": len(CLASS_NAMES),
    "names": CLASS_NAMES,
}, data_yaml.open("w"), sort_keys=False)
print("\nwrote", data_yaml); print(data_yaml.read_text())

> **Watch the indicator count.** It should be close to the image count — roughly one per
> frame. Far fewer means many frames were labelled without it, and the model will learn
> that the indicator is frequently absent, which directly depresses the recall that
> matters most.

## 4. Split leakage check

Frames pulled from video are near-duplicates of their neighbours. If frame 41 is in
`train` and frame 42 in `val`, validation mAP measures memorisation, not generalisation —
and it will look excellent.

This compares a downscaled perceptual hash of every image across splits. Any exact or
near-exact match spanning two splits is leakage.

In [ ]:
import numpy as np
from PIL import Image

def dhash(path, size=8):
    """64-bit difference hash — robust to compression, sensitive to content."""
    img = Image.open(path).convert("L").resize((size + 1, size), Image.LANCZOS)
    a = np.asarray(img, dtype=np.int16)
    bits = (a[:, 1:] > a[:, :-1]).flatten()
    return int("".join("1" if b else "0" for b in bits), 2)

hashes = {}
for split in (TRAIN_SPLIT, VAL_SPLIT, TEST_SPLIT):
    hashes[split] = {}
    for im in list_images(split):
        try:
            hashes[split][im.name] = dhash(im)
        except Exception as exc:
            print("hash failed:", im.name, exc)
    print(f"hashed {len(hashes[split]):>4} in {split}")

def hamming(a, b): return bin(a ^ b).count("1")

NEAR = 3   # <=3 differing bits => visually near-identical
leaks = []
splits = [TRAIN_SPLIT, VAL_SPLIT, TEST_SPLIT]
for i, s1 in enumerate(splits):
    for s2 in splits[i + 1:]:
        for n1, h1 in hashes[s1].items():
            for n2, h2 in hashes[s2].items():
                d = hamming(h1, h2)
                if d <= NEAR:
                    leaks.append((s1, n1, s2, n2, d))

print(f"\nnear-duplicate pairs across splits: {len(leaks)}")
for row in leaks[:20]:
    print(f"  {row[0]}/{row[1]}  <->  {row[2]}/{row[3]}   hamming={row[4]}")
if leaks:
    print("\n*** LEAKAGE. Re-split by CLIP/SESSION, not by frame. Val metrics below are optimistic. ***")
else:
    print("\nNo cross-split near-duplicates detected.")

## 5. Box-size distributions

Two uses:

1. Confirms the indicator really is tiny, which is what justifies the P2 head.
2. `player_w`, `player_h`, `player_dw`, `player_dh` are **observation features** in the RL
   state. Their normalisation ranges should come from measurement, not from a guess.

In [ ]:
import matplotlib.pyplot as plt

boxes = {c: {"w": [], "h": [], "area_px": []} for c in range(len(CLASS_NAMES))}
REF_W, REF_H = 1920, 1080

for split in (TRAIN_SPLIT, VAL_SPLIT):
    for im in list_images(split):
        for cls, cx, cy, w, h in read_label(label_for(im, split)):
            if cls in boxes:
                boxes[cls]["w"].append(w)
                boxes[cls]["h"].append(h)
                boxes[cls]["area_px"].append((w * REF_W) * (h * REF_H))

print(f"{'class':<16}{'n':>6}{'w_med':>9}{'h_med':>9}{'w_p99':>9}{'h_p99':>9}{'px@1080p':>12}")
for cls, name in enumerate(CLASS_NAMES):
    w, h, a = (np.array(boxes[cls][k]) for k in ("w", "h", "area_px"))
    if len(w) == 0:
        print(f"{name:<16}{0:>6}"); continue
    side = np.sqrt(a)
    print(f"{name:<16}{len(w):>6}{np.median(w):>9.4f}{np.median(h):>9.4f}"
          f"{np.percentile(w,99):>9.4f}{np.percentile(h,99):>9.4f}{np.median(side):>11.1f}px")

fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
for ax, cls in zip(axes, range(len(CLASS_NAMES))):
    a = np.array(boxes[cls]["area_px"])
    if len(a):
        ax.hist(np.sqrt(a), bins=40, color="#2a78d6")
        ax.axvline(np.median(np.sqrt(a)), color="#eb6834", lw=2)
    ax.set_title(f"{CLASS_NAMES[cls]} — sqrt(area) px @1080p")
    ax.set_xlabel("px"); ax.grid(alpha=0.25)
plt.tight_layout(); plt.show()

ind_side = np.sqrt(np.array(boxes[INDICATOR_ID]["area_px"]))
if len(ind_side):
    at_imgsz = np.median(ind_side) * (IMGSZ / REF_W)
    print(f"\nindicator median: {np.median(ind_side):.1f}px @1080p -> ~{at_imgsz:.1f}px at imgsz={IMGSZ}")
    print("P3 stride is 8 -> that is under 2 feature cells. P2 (stride 4) is justified."
          if at_imgsz < 24 else "Indicator is large enough that P2 may be optional; benchmark both.")

## 6. Train

`yolo26s-p2.yaml` may not ship in every Ultralytics release. The cell falls back to the
plain config and reports it clearly rather than failing silently — a P2-less run is still
useful, it just changes how to read the indicator recall.

In [ ]:
from ultralytics import YOLO

def build_model(cfg, pretrained):
    try:
        m = YOLO(cfg)
        if pretrained:
            try:
                m.load(pretrained)
                print(f"built {cfg} + transferred weights from {pretrained}")
            except Exception as exc:
                print(f"built {cfg} from scratch (weight transfer failed: {exc})")
        return m, cfg
    except Exception as exc:
        fallback = cfg.replace("-p2", "")
        print(f"!! {cfg} unavailable ({exc})\n!! falling back to {fallback} — NO P2 HEAD")
        return YOLO(fallback), fallback

model, used_cfg = build_model(MODEL_CFG, PRETRAINED)
USED_P2 = "p2" in used_cfg.lower()
print("P2 head:", USED_P2)

In [ ]:
results = model.train(
    data=str(data_yaml),
    imgsz=IMGSZ,
    epochs=EPOCHS,
    patience=PATIENCE,
    batch=BATCH,
    seed=SEED,
    project=PROJECT,
    name=NAME,
    exist_ok=True,
    pretrained=True,
    optimizer="auto",
    cos_lr=True,
    close_mosaic=25,      # mosaic hurts small-object localisation late on
    # augmentation: all three classes are horizontally symmetric, so fliplr is free data
    fliplr=0.5,
    flipud=0.0,           # the game is never upside down
    degrees=0.0,          # nor rotated
    scale=0.5,
    translate=0.1,
    mosaic=1.0,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    plots=True,
    val=True,
)
best_pt = Path(results.save_dir) / "weights" / "best.pt"
print("\nbest weights:", best_pt)

## 7. Per-class metrics — the indicator recall gate

Overall mAP is dominated by `character`, which is large and easy. It will look fine even
if the indicator is being missed constantly.

**The number that matters is `indicator_self` recall**, because a miss means agent
identity is unknown for that frame and every relational feature is wrong.

In [ ]:
best = YOLO(str(best_pt))
metrics = best.val(data=str(data_yaml), imgsz=IMGSZ, split="val", plots=False)

per_class = {}
print(f"{'class':<16}{'P':>9}{'R':>9}{'mAP50':>9}{'mAP50-95':>11}")
for i, name in enumerate(CLASS_NAMES):
    try:
        p, r, ap50, ap = metrics.box.class_result(i)
    except Exception:
        p = r = ap50 = ap = float("nan")
    per_class[name] = {"precision": float(p), "recall": float(r),
                       "map50": float(ap50), "map50_95": float(ap)}
    print(f"{name:<16}{p:>9.4f}{r:>9.4f}{ap50:>9.4f}{ap:>11.4f}")
print(f"\n{'ALL':<16}{'':>9}{'':>9}{metrics.box.map50:>9.4f}{metrics.box.map:>11.4f}")

ind_recall = per_class["indicator_self"]["recall"]

# With ~800 images the val split holds only ~80 indicators, so the point estimate is
# noisy: 0.96 and 0.98 can be the same underlying rate. Wilson interval, because the
# normal approximation is unreliable for proportions this close to 1.
n_ind_val = inventory[VAL_SPLIT]["counts"][INDICATOR_ID]
def wilson(p, n, z=1.96):
    if n == 0:
        return (float("nan"), float("nan"))
    d = 1 + z*z/n
    c = (p + z*z/(2*n)) / d
    h = z*np.sqrt(p*(1-p)/n + z*z/(4*n*n)) / d
    return (max(0.0, c-h), min(1.0, c+h))
lo, hi = wilson(ind_recall, n_ind_val)

print("\n" + "=" * 74)
print(f"INDICATOR RECALL: {ind_recall:.4f}   95% CI [{lo:.4f}, {hi:.4f}]  (n={n_ind_val})")
print("=" * 74)
if n_ind_val < 200:
    print(f"NOTE: only {n_ind_val} val indicators. The interval spans "
          f"{(hi-lo)*100:.1f} points, so treat the gate below as indicative, not decisive.\n")
report_ci = {"n": int(n_ind_val), "ci_low": float(lo), "ci_high": float(hi)}
if ind_recall >= INDICATOR_RECALL_STRONG:
    verdict = ("STRONG. Carry-forward fallback in detection_schema.py is sufficient as-is. "
               "Roughly 1 frame in 50 or better needs it.")
elif ind_recall >= INDICATOR_RECALL_USABLE:
    verdict = ("USABLE. Add a staleness decay: the longer identity has been carried forward, "
               "the lower `player_confidence` should read, so the policy learns to be cautious.")
else:
    misses = (1 - ind_recall) * 40
    verdict = (f"TOO LOW. At 40 Hz this is ~{misses:.1f} unidentified frames per second. "
               "The indicator cannot be the sole identity source. Options: raise imgsz, "
               "confirm P2 is active, label the triangle with more margin, or add a second cue.")
print(verdict)

report = {"config": used_cfg, "p2": USED_P2, "imgsz": IMGSZ,
          "per_class": per_class, "map50": float(metrics.box.map50),
          "map50_95": float(metrics.box.map), "indicator_recall": ind_recall,
          "indicator_recall_ci": report_ci, "verdict": verdict}

## 8. Indicator→character geometry — calibrating the association constants

`feature_extractor/memory/detection_schema.py` matches the indicator to a character with

```python
score = INDICATOR_HORIZONTAL_WEIGHT * |dx| + INDICATOR_VERTICAL_WEIGHT * max(0, dy)
```

and rejects a match when `score > INDICATOR_MAX_MATCH_SCORE`, or when the character sits
more than `INDICATOR_ABOVE_TOLERANCE` above the indicator.

Those three constants are currently **guesses about how far above the head the triangle
floats**. This section measures them from ground truth and emits a block to paste in.

Method: for every labelled frame with an indicator and at least one character, the true
pair is the character nearest below the indicator. Its `(dx, dy)` gives the true-pair
distribution; every other character in the frame gives the false-pair distribution. A
usable threshold must separate the two.

In [ ]:
true_dx, true_dy, true_scores, false_scores = [], [], [], []
frames_multi_char = 0

H_W, V_W = 2.0, 1.0   # weights from detection_schema.py

for split in (TRAIN_SPLIT, VAL_SPLIT):
    for im in list_images(split):
        rows = read_label(label_for(im, split))
        inds  = [r for r in rows if r[0] == INDICATOR_ID]
        chars = [r for r in rows if r[0] == CHARACTER_ID]
        if len(inds) != 1 or not chars:
            continue
        if len(chars) > 1:
            frames_multi_char += 1
        _, ix, iy, _, _ = inds[0]

        below = [c for c in chars if c[2] >= iy]
        pool = below or chars
        match = min(pool, key=lambda c: abs(c[1] - ix) + abs(c[2] - iy))

        for c in chars:
            dx, dy = c[1] - ix, c[2] - iy
            score = H_W * abs(dx) + V_W * max(0.0, dy)
            if c is match:
                true_dx.append(dx); true_dy.append(dy); true_scores.append(score)
            else:
                false_scores.append(score)

true_dx = np.array(true_dx); true_dy = np.array(true_dy)
true_scores = np.array(true_scores); false_scores = np.array(false_scores)

print(f"true pairs: {len(true_scores)}   false pairs: {len(false_scores)}"
      f"   (frames with >1 character: {frames_multi_char})")
print("\n-- offset from indicator centre to its character centre (normalised) --")
for label, arr in (("dx", true_dx), ("dy", true_dy)):
    print(f"  {label}: min={arr.min():+.4f}  p1={np.percentile(arr,1):+.4f}  "
          f"med={np.median(arr):+.4f}  p99={np.percentile(arr,99):+.4f}  max={arr.max():+.4f}")

print("\n-- match score --")
print(f"  TRUE : med={np.median(true_scores):.4f}  p99={np.percentile(true_scores,99):.4f}  max={true_scores.max():.4f}")
if len(false_scores):
    print(f"  FALSE: min={false_scores.min():.4f}  p1={np.percentile(false_scores,1):.4f}  med={np.median(false_scores):.4f}")
    gap = false_scores.min() - true_scores.max()
    print(f"\n  separation gap (min false - max true): {gap:+.4f}")
    print("  -> cleanly separable" if gap > 0 else
          "  -> OVERLAP: some wrong pairings score better than correct ones. "
          "Widen the margin or add a cue; expect occasional identity errors.")
else:
    print("  FALSE: none (no multi-character frames) — cannot verify separability.")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
axes[0].hist(true_dx, bins=40, color="#2a78d6"); axes[0].set_title("dx (indicator -> character)")
axes[1].hist(true_dy, bins=40, color="#1baf7a"); axes[1].set_title("dy (positive = character below)")
axes[2].hist(true_scores, bins=40, alpha=0.85, label="true pair", color="#2a78d6")
if len(false_scores):
    axes[2].hist(false_scores, bins=40, alpha=0.6, label="false pair", color="#e34948")
axes[2].set_title("match score"); axes[2].legend()
for ax in axes: ax.grid(alpha=0.25)
plt.tight_layout(); plt.show()

In [ ]:
# Threshold sits above the worst true pair with margin, and below the best false pair.
true_p999 = float(np.percentile(true_scores, 99.9))
max_score = true_p999 * 1.25
if len(false_scores):
    max_score = min(max_score, float(false_scores.min()) * 0.9)
above_tol = float(max(0.005, abs(min(0.0, np.percentile(true_dy, 0.5))) * 1.5))

block = f"""# ---- measured on {len(true_scores)} ground-truth pairs; replaces the initial estimates ----
INDICATOR_HORIZONTAL_WEIGHT = {H_W}
INDICATOR_VERTICAL_WEIGHT   = {V_W}
INDICATOR_MAX_MATCH_SCORE   = {max_score:.4f}   # was 0.22 (guess)
INDICATOR_ABOVE_TOLERANCE   = {above_tol:.4f}   # was 0.02 (guess)"""

print(block)
print("\nPaste into feature_extractor/memory/detection_schema.py")
report["indicator_geometry"] = {
    "n_true_pairs": int(len(true_scores)), "n_false_pairs": int(len(false_scores)),
    "dx_median": float(np.median(true_dx)), "dy_median": float(np.median(true_dy)),
    "true_score_p999": true_p999,
    "false_score_min": float(false_scores.min()) if len(false_scores) else None,
    "INDICATOR_MAX_MATCH_SCORE": round(max_score, 4),
    "INDICATOR_ABOVE_TOLERANCE": round(above_tol, 4),
}
(OUT / "detection_schema_constants.py").write_text(block)

## 9. The overlap case

Nearest-to-last-position association fails when two fighters overlap — this is exactly
what the indicator is for. This measures how often the hard case occurs and whether
predictions still resolve it correctly.

In [ ]:
def iou(a, b):
    ax1, ay1, ax2, ay2 = a[1]-a[3]/2, a[2]-a[4]/2, a[1]+a[3]/2, a[2]+a[4]/2
    bx1, by1, bx2, by2 = b[1]-b[3]/2, b[2]-b[4]/2, b[1]+b[3]/2, b[2]+b[4]/2
    ix, iy = max(0, min(ax2,bx2)-max(ax1,bx1)), max(0, min(ay2,by2)-max(ay1,by1))
    inter = ix*iy
    union = a[3]*a[4] + b[3]*b[4] - inter
    return inter/union if union > 0 else 0.0

overlap_frames, resolved, unresolved = [], 0, 0
for im in list_images(VAL_SPLIT):
    rows = read_label(label_for(im, VAL_SPLIT))
    chars = [r for r in rows if r[0] == CHARACTER_ID]
    if len(chars) < 2:
        continue
    if max(iou(a, b) for i, a in enumerate(chars) for b in chars[i+1:]) > 0.10:
        overlap_frames.append(im)

print(f"val frames with overlapping characters: {len(overlap_frames)} / {len(list_images(VAL_SPLIT))}")

for im in overlap_frames:
    pred = best.predict(str(im), imgsz=IMGSZ, conf=0.25, verbose=False)[0]
    cls = pred.boxes.cls.cpu().numpy().astype(int)
    if (cls == INDICATOR_ID).sum() == 1 and (cls == CHARACTER_ID).sum() >= 2:
        resolved += 1
    else:
        unresolved += 1

total = resolved + unresolved
if total:
    print(f"indicator present and >=2 characters detected: {resolved}/{total} ({resolved/total:.1%})")
    print("The remainder are frames where identity would fall back to carry-forward "
          "in exactly the situation carry-forward is least reliable.")
    report["overlap"] = {"frames": len(overlap_frames), "resolved": resolved,
                         "resolution_rate": resolved/total}
else:
    print("No overlapping-character frames in val. Add some — this is the failure case that matters.")

## 10. Latency benchmark

In [ ]:
import time

print("Kaggle GPU is a rough proxy for the 3050 Ti; treat these as relative, not absolute.\n")
print(f"{'imgsz':>7}{'ms/frame':>11}{'FPS':>9}{'40Hz budget':>14}")
for size in (640, 768, 960, 1280):
    dummy = torch.zeros(1, 3, size, size)
    for _ in range(5):
        best.predict(dummy, imgsz=size, verbose=False)
    torch.cuda.synchronize(); t0 = time.perf_counter()
    for _ in range(30):
        best.predict(dummy, imgsz=size, verbose=False)
    torch.cuda.synchronize()
    ms = (time.perf_counter() - t0) / 30 * 1000
    ok = "OK" if ms < 15 else ("TIGHT" if ms < 25 else "TOO SLOW")
    print(f"{size:>7}{ms:>11.2f}{1000/ms:>9.1f}{ok:>14}")

print("\nA 40 Hz control loop leaves 25 ms per step for capture + detect + state + policy.")
print("Detection should stay under ~15 ms. If it does not, drop model size before dropping imgsz:")
print("imgsz is what makes the indicator detectable at all.")

## 11. Export

**Do not export the TensorRT engine here.** Engines are not portable across GPU
architecture, driver version, or TensorRT version — a Kaggle T4/P100 engine will not load
on the 3050 Ti. Export `.pt` and `.onnx`, then build the engine locally.

In [ ]:
best.export(format="onnx", imgsz=IMGSZ, opset=17, simplify=True, dynamic=False)

import shutil, json
final_dir = OUT / "export"; final_dir.mkdir(exist_ok=True)
for f in (Path(best_pt).parent).glob("best.*"):
    shutil.copy(f, final_dir / f.name)

report["export"] = {"imgsz": IMGSZ, "files": [f.name for f in final_dir.iterdir()]}
(OUT / "detector_report.json").write_text(json.dumps(report, indent=2))

print("exported to", final_dir)
for f in sorted(final_dir.iterdir()):
    print(f"  {f.name:<24}{f.stat().st_size/1e6:>8.1f} MB")
print("\nreport ->", OUT / "detector_report.json")

### On your laptop

```powershell
C:\venvs\brawl312\Scripts\Activate.ps1

# 1. build the engine on the target GPU
python -c "from ultralytics import YOLO; YOLO('feature_extractor/yolo/best.pt').export(format='engine', half=True, imgsz=960, device=0)"

# 2. match inference resolution to training resolution in extract.py
#    infer_width / infer_height must correspond to imgsz=960

# 3. delete the grayscale conversion at extract.py:72-73 --
#    it will destroy accuracy on a colour-trained model

python -m pytest tests -q
python tools/debug_observation_overlay.py --phase weapon_acquisition --show --max-steps 600
```

`Extract` reads class names from model metadata, so `['character','indicator_self','weapon']`
carries through with no code change.

## Appendix — troubleshooting

### `CUDA error: no kernel image is available for execution on the device`

Raised at `model.to(device)`, usually right after `Transferred N/N items from pretrained
weights`. It means the torch binary contains no compiled kernels for this GPU's compute
capability. Confirm with:

```python
import torch
print(torch.cuda.get_device_capability(0))   # e.g. (6, 0) -> sm_60
print(torch.cuda.get_arch_list())            # e.g. ['sm_75','sm_80','sm_86','sm_90']
```

If the device arch is absent from the list, that is the cause. Almost always it is a pip
install having upgraded torch underneath you.

| Fix | How |
|---|---|
| **Switch GPU** (preferred) | Settings → Accelerator → **GPU T4 x2**. T4 is `sm_75`; P100 is `sm_60`, which recent torch no longer builds for |
| **Stop upgrading torch** | Install with `--no-deps` (section 2 now does this) |
| **Revert torch** | `!pip install -q --force-reinstall torch==<original> torchvision==<original>` then **Run → Restart session** |

Restarting the session is required after any torch reinstall — the old module stays
loaded in memory otherwise.

### `CUDA out of memory` at `imgsz=960`

Lower `BATCH` to 8, then 4. A P2 head at 960 is memory-hungry. Prefer reducing batch over
reducing `imgsz`: resolution is what makes the indicator detectable at all.

### `yolo26s-p2.yaml` not found

Section 6 falls back to the non-P2 config and prints a warning. A P2-less run still
trains; it just changes how to read the indicator recall, since the small-object head is
the thing that recall depends on.

## 12. Summary

Report these back:

| Item | Where | Why |
|---|---|---|
| `indicator_self` recall | §7 | Sets the identity-fallback design |
| Separation gap | §8 | If negative, the indicator scheme needs a second cue |
| `INDICATOR_MAX_MATCH_SCORE`, `INDICATOR_ABOVE_TOLERANCE` | §8 | Replaces guessed constants |
| Box-size percentiles | §5 | Normalisation ranges for `player_w/h` |
| Overlap resolution rate | §9 | The case positional association cannot handle |
| Latency at chosen imgsz | §10 | Decides `s` vs `m` |
| Leakage pairs | §4 | If non-zero, every metric above is optimistic |

`/kaggle/working/detector_report.json` contains all of it in machine-readable form.

In [ ]:
import json
print(json.dumps(report, indent=2))